# Practice Assignment VC (bonus week): Vibe-code it, then prove it
# تمرین VC (هفته‌ی اضافه): کد را با هوش مصنوعی بنویسید، سپس درستی‌اش را ثابت کنید

**Course:** AI for Aerospace & Mechanical Engineering (PhD), bonus week "Vibe Coding in
Engineering" &nbsp;|&nbsp; **Effort:** about 4 hours &nbsp;|&nbsp; **Weight:** formative
(feedback only; the bonus week is outside the 16-week grading plan)

> **Provenance:** new content. Code verification with exact (manufactured) solutions and the
> observed order of accuracy after Veluri, Roy and Luke, *Computers & Fluids* (2012); mutation
> testing after DeMillo, Lipton and Sayward, *IEEE Computer* 11(4) (1978); "count what you asked
> for" and the deleted-tests story paraphrased from Kim and Yegge, *Vibe Coding* (IT Revolution,
> 2025), Chs. 4 and 7; package hallucination after Spracklen et al., USENIX Security 2025.
> The static-fire log is synthetic (`tools/make_week_extra_vibe_coding_data.py`).

<div dir="rtl">

## صورت مسئله

یک موتور سوخت جامد کوچک روی سکوی آزمون بسته شده و یک لودسل نیروی پیشران را ثبت کرده است. فایل <code>data/sample/practice_vc_static_fire.csv</code> دو ستون دارد: زمان (ثانیه) و نیروی پیشران (نیوتن). این فایل **مصنوعی** است (یک پالس نیم‌سینوسی با نویز)، بنابراین پاسخ درست آن را می‌دانیم. سه ویژگی لاگ مهم است: فاصله‌ی زمانی نمونه‌ها **یکنواخت نیست** (لرزش زمانی و بسته‌های گم‌شده)، پیش و پس از سوختن لودسل مقادیر **منفی** کوچک (نویز) نشان می‌دهد، و جرم پیشرانه ۰٫۰۶۵ کیلوگرم است.

این بار شما کد را خودتان نمی‌نویسید: آن را از یک دستیار هوش مصنوعی (هر ابزاری که در دسترس دارید) می‌خواهید. کار اصلی شما **راستی‌آزمایی** است:

۱. **درخواست و ثبت:** فایل‌های <code>SPEC.md</code> و <code>thrust_analysis.py</code> را به دستیار بدهید. هر درخواست را دقیقاً در <code>prompt_log.md</code> ثبت کنید و نخستین پاسخ کامل را **بدون هیچ تغییری** در <code>ai_raw_output.py</code> بگذارید.

۲. **بررسی ایستا:** پیش از اجرای کد، بشمارید که همه‌ی اجزای خواسته‌شده تحویل شده‌اند و هیچ بسته‌ی ناشناخته‌ای import نشده است.

۳. **راستی‌آزمایی:** پاسخ خام را با مقادیر مرجعِ محاسبه‌شده با دست، ویژگی‌ها (properties) و **مرتبه‌ی همگرایی** روی یک جواب ساختگیِ دقیق (نیم‌سینوس) بسنجید. هر نقص را بازتولید کنید.

۴. **نسخه‌ی نهایی:** کد بازبینی‌شده و اصلاح‌شده را در <code>thrust_analysis.py</code> بگذارید.

۵. **آزمون‌های خودتان:** دست‌کم ۸ آزمون pytest بنویسید و با «جهش‌یافته‌ها» (mutants) نشان دهید که آزمون‌هایتان خطاهای محتمل را می‌گیرند.

۶. **گزارش:** جدول نقص‌ها و پاسخ سه پرسش تأملی را در <code>REVIEW.md</code> بنویسید.

اگر پاسخ دستیار از ابتدا بی‌نقص بود، همین را با شواهد بنویسید؛ نمره‌ی شما به کیفیت راستی‌آزمایی بستگی دارد، نه به تعداد نقص‌ها. هرگز داده یا کدی را که آزمایشگاه یا کارفرما اجازه‌ی اشتراکش را نداده، در ابزار بیرونی وارد نکنید. اگر به هیچ دستیار هوش مصنوعی دسترسی ندارید، از مدرس یک پاسخ آماده بگیرید.

</div>

## Setup (run first, do not edit)

This notebook never needs internet access. It loads your files from this folder; the helper
`load_module` executes a Python file, so **read `ai_raw_output.py` before you run Section 3**.

In [ ]:
from __future__ import annotations

import importlib.util
import math
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path
from collections.abc import Callable, Sequence
from types import ModuleType
from typing import Final

import numpy as np


def find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the folder that contains ``src/aerospace_ai``."""
    for folder in (start, *start.parents):
        if (folder / "src" / "aerospace_ai").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from inside the course repository.")


REPO_ROOT: Final[Path] = find_repo_root(Path.cwd())
HERE: Final[Path] = REPO_ROOT / "assignments" / "practice_vibe_coding"
for folder in (REPO_ROOT / "src", HERE):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from aerospace_ai.verification import (  # noqa: E402
    ConvergenceCheck,
    Mutant,
    PropertyCheck,
    ReferenceCase,
    ReferenceCheck,
    Verifier,
    audit_deliverables,
    unresolvable_imports,
)

LOG_CSV: Final[Path] = REPO_ROOT / "data" / "sample" / "practice_vc_static_fire.csv"
PROPELLANT_MASS_KG: Final[float] = 0.065
REQUESTED: Final[tuple[str, ...]] = ("ThrustCurve", "specific_impulse")
PYTEST_FLAGS: Final[tuple[str, ...]] = (
    "-q",
    "--color=no",
    "-p",
    "no:warnings",
    "-p",
    "no:cacheprovider",
)


def load_module(path: Path, name: str) -> ModuleType | None:
    """Import a Python file as a module; return None if it is empty or does not import."""
    if not path.exists() or "class ThrustCurve" not in path.read_text(encoding="utf-8"):
        print(f"[ ] {path.name}: no ThrustCurve class yet")
        return None
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(module)
    except Exception as error:  # noqa: BLE001 - a broken file is a finding, not a crash
        print(f"[!] {path.name} does not import: {type(error).__name__}: {error}")
        return None
    return module


def run_suite(module_source: str, tests: Path) -> bool:
    """Run the pytest file ``tests`` against ``module_source`` (as thrust_analysis.py)."""
    with tempfile.TemporaryDirectory() as tmp:
        work = Path(tmp)
        (work / "thrust_analysis.py").write_text(module_source, encoding="utf-8")
        shutil.copy(tests, work / tests.name)
        command = [
            sys.executable,
            "-m",
            "pytest",
            str(work / tests.name),
            *PYTEST_FLAGS,
        ]
        env_path = str(REPO_ROOT / "src")
        result = subprocess.run(
            [*command, f"--rootdir={work}"],
            cwd=REPO_ROOT,
            env={**os.environ, "PYTHONPATH": env_path},
            capture_output=True,
            text=True,
        )
    return result.returncode == 0


print("Repository root found:", REPO_ROOT.name)

## Task 1 (10 points): record the prompt and the raw answer

Fill in `prompt_log.md` and paste the assistant's first complete answer into
`ai_raw_output.py` without editing it. The cell below only checks that the files are not empty.

In [ ]:
raw_path = HERE / "ai_raw_output.py"
raw_source = raw_path.read_text(encoding="utf-8")
log_rows = [
    line
    for line in (HERE / "prompt_log.md").read_text(encoding="utf-8").splitlines()
    if line.startswith("| 1 |")
]
has_raw = "class ThrustCurve" in raw_source
has_log = bool(log_rows) and len(log_rows[0].replace("|", "").split()) > 1
print(f"[{'x' if has_raw else ' '}] ai_raw_output.py contains an answer")
print(f"[{'x' if has_log else ' '}] prompt_log.md entry 1 is filled in")

## Task 2 (part of Task 5's 20 points): static checks before running anything

"Count what you asked for": an assistant can return less than you requested, or quietly add
things you did not ask for. And it can import packages that do not exist; attackers register
such names. Both checks below read the source text and execute nothing.

In [ ]:
if has_raw:
    audit = audit_deliverables(raw_source, REQUESTED)
    print("missing:", sorted(audit.missing) or "nothing")
    print("not requested (review these too):", sorted(audit.unrequested) or "nothing")
    print(
        "imports not resolvable here:",
        sorted(unresolvable_imports(raw_source)) or "none",
    )
else:
    print("[ ] paste the raw answer first")

## Task 3 (35 points, hidden tests): verify, then fix

**Exercise 3.1 (by hand, write the numbers in `REVIEW.md`).** Compute the exact impulse of:
(a) a triangle pulse 0 → 10 N → 0 over 2 s; (b) a constant 4 N on the uneven grid
t = 0, 0.1, 0.5, 3.0 s; (c) the pulse in (a) preceded by one noise sample of −2 N at t = −1 s.

**Exercise 3.2.** Complete the verifier: the reference cases of 3.1, one property, and a
convergence check on the manufactured half-sine (the trapezoidal rule is second order, so the
observed order must approach 2). Then run it on the raw answer and on your final file.

In [ ]:
PEAK_N: Final[float] = 120.0
BURN_S: Final[float] = 1.6
EXACT_HALF_SINE_NS: Final[float] = 2.0 * PEAK_N * BURN_S / math.pi
STEP_SIZES: Final[tuple[float, ...]] = (BURN_S / 20, BURN_S / 40, BURN_S / 80)
Impulse = Callable[[Sequence[float], Sequence[float]], float]


def impulse_function(module: ModuleType) -> Impulse:
    """Wrap a module's ThrustCurve as ``impulse(time, thrust) -> N s`` for the checks."""

    def impulse(time: Sequence[float], thrust: Sequence[float]) -> float:
        return module.ThrustCurve(
            np.asarray(time, float), np.asarray(thrust, float)
        ).total_impulse()

    return impulse


def half_sine_error(impulse: Impulse, h: float) -> float:
    """Error on the manufactured half-sine pulse sampled with step ``h``."""
    t = np.linspace(0.0, BURN_S, round(BURN_S / h) + 1)
    return abs(impulse(t, PEAK_N * np.sin(np.pi * t / BURN_S)) - EXACT_HALF_SINE_NS)


def scale_sample(rng: np.random.Generator) -> tuple[float]:
    """A random positive scale factor for the linearity property."""
    return (float(rng.uniform(0.1, 10.0)),)


def is_linear(impulse: Impulse, args: tuple[float]) -> bool:
    """Scaling the thrust by s scales the impulse by s (for non-negative thrust)."""
    time, thrust = [0.0, 0.3, 1.0], np.array([2.0, 5.0, 1.0])
    return math.isclose(
        impulse(time, args[0] * thrust), args[0] * impulse(time, thrust), rel_tol=1e-9
    )

In [ ]:
reference_cases = [
    ReferenceCase(
        "triangle", ([0.0, 1.0, 2.0], [0.0, 10.0, 0.0]), 10.0, source="hand, 3.1a"
    ),
    # TODO: add the cases of Exercise 3.1 b) and c) with your hand-computed values.
]
linear_in_thrust = (
    None  # TODO: PropertyCheck("impulse is linear in thrust", scale_sample, is_linear)
)
verifier = None  # TODO: Verifier([ReferenceCheck(...), linear_in_thrust, ConvergenceCheck(...)])

In [ ]:
candidates = {}
raw_module = load_module(raw_path, "ai_raw_output") if has_raw else None
final_module = load_module(HERE / "thrust_analysis.py", "thrust_analysis_final")
if raw_module is not None:
    candidates["AI raw answer"] = raw_module
if final_module is not None:
    try:
        final_module.ThrustCurve(np.array([0.0, 1.0]), np.array([0.0, 1.0]))
        candidates["your final file"] = final_module
    except NotImplementedError:
        print("[ ] thrust_analysis.py: still the skeleton")
if verifier is not None and candidates:
    display(verifier.compare({k: impulse_function(m) for k, m in candidates.items()}))
else:
    print("[ ] nothing to verify yet")

**Expected output** of your final file on the sample log (the hidden tests use these):

| Quantity | Expected |
|---|---|
| samples | 498 |
| total impulse | 122.296 N s (the noise-free pulse has 122.231 N s) |
| peak thrust | 120.508 N |
| burn time (≥ 5 % of peak) | 1.5408 s |
| average thrust | 79.37 N |
| specific impulse (m_p = 0.065 kg) | 191.86 s |

In [ ]:
if "your final file" in candidates:
    curve = final_module.ThrustCurve.from_csv(LOG_CSV)
    impulse = curve.total_impulse()
    print(
        f"samples {curve.time_s.size}, impulse {impulse:.3f} N s, peak {curve.peak_thrust():.3f} N, "
        f"burn {curve.burn_time():.4f} s, average {curve.average_thrust():.2f} N, "
        f"Isp {final_module.specific_impulse(impulse, PROPELLANT_MASS_KG):.2f} s"
    )
    assert math.isclose(
        impulse, 122.2955, abs_tol=1e-3
    ), "total impulse differs from the expected value"
else:
    print("[ ] final file not ready")

## Task 4 (25 points): your own tests, and proof that they bite

Write at least 8 tests in `test_thrust_analysis.py` (fixture, `parametrize` and `raises` each
used at least once; one expected value computed by hand). A suite that passes on correct code
is not enough: it must **fail** on wrong code. Define at least three mutants of *your own*
final file below, one per likely defect class (spacing, clipping, threshold, units, input
validation). Each `find` string must appear exactly once in your file. The instructor grades
your suite against six hidden mutants.

In [ ]:
my_mutants = [
    # TODO: Mutant("name", "text that appears once in YOUR thrust_analysis.py", "broken text", "why")
]

In [ ]:
tests_path = HERE / "test_thrust_analysis.py"
final_source = (HERE / "thrust_analysis.py").read_text(encoding="utf-8")
if "your final file" not in candidates or not my_mutants:
    print("[ ] final file or mutants not ready")
elif not run_suite(final_source, tests_path):
    print(
        "[!] your suite fails on your own final file: fix the code or the tests first"
    )
else:
    killed = []
    for mutant in my_mutants:
        try:
            mutated = mutant.apply(final_source)
        except ValueError as error:
            print(f"[!] {mutant.name}: {error}")
            continue
        if not run_suite(mutated, tests_path):
            killed.append(mutant.name)
        print(
            f"{'killed  ' if mutant.name in killed else 'SURVIVED'} {mutant.name}: {mutant.description}"
        )
    print(f"mutation score {len(killed)}/{len(my_mutants)}")

## Task 5 (20 points) and reflection (10 points)

Complete `REVIEW.md`: one row per defect you reproduced (input, check that found it, fix), the
checks you ran, and the three reflection answers. Then run the cell below one last time.

In [ ]:
review = (HERE / "REVIEW.md").read_text(encoding="utf-8")
n_tests = (
    (HERE / "test_thrust_analysis.py").read_text(encoding="utf-8").count("\ndef test_")
)
print(f"[{'x' if n_tests >= 8 else ' '}] {n_tests} tests written (at least 8)")
print(f"[{'x' if '| 1 | |' not in review else ' '}] REVIEW.md defect table started")
print(
    f"[{'x' if 'TODO' not in final_source else ' '}] no TODO left in thrust_analysis.py"
)